# Task 1: Exploratory Data Analysis
NYC Airbnb Price Prediction — DSC 148

**Goal:** Understand distributions, missingness, and correlations before building the preprocessing pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 60)
plt.rcParams['figure.figsize'] = (10, 4)

DATA_DIR = '../raw files/'

train = pd.read_csv(DATA_DIR + 'train.csv', low_memory=False)
test  = pd.read_csv(DATA_DIR + 'test.csv',  low_memory=False)

print('Train:', train.shape)
print('Test: ', test.shape)

## 1. Basic Overview

In [ ]:
train.head(3)

In [ ]:
train.dtypes.value_counts()

In [ ]:
# Columns that exist in train but not test (should only be 'price')
only_in_train = set(train.columns) - set(test.columns)
print('Only in train:', only_in_train)

## 2. Target Variable: price

In [ ]:
price = train['price']

print('Summary statistics:')
print(price.describe())
print(f'\nZero-price listings: {(price == 0).sum()}')
print(f'Listings over $500:  {(price > 500).sum()}')
print(f'Listings over $1000: {(price > 1000).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(price.clip(0, 600), bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (clipped at $600)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(price), bins=80, color='seagreen', edgecolor='white')
axes[1].set_title('log1p(Price) Distribution')
axes[1].set_xlabel('log1p(Price)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print('\nlog1p(price) is much more normal — use this as training target.')

## 3. Missing Value Analysis

In [ ]:
missing = train.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(1)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['missing'] > 0]

print(f'{len(missing_df)} columns have missing values:\n')
print(missing_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
missing_df['pct'].plot(kind='barh', ax=ax, color='tomato')
ax.axvline(50, color='black', linestyle='--', linewidth=0.8, label='50%')
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Value Rate by Column')
ax.legend()
plt.tight_layout()
plt.show()

print('\nKey findings:')
print('  host_acceptance_rate: 100% missing — DROP')
print('  square_feet: 99% missing  — add missingness flag, impute with 0')
print('  review_scores_*: ~23% missing — missing = listing has no reviews (informative)')

## 4. Numeric Feature Correlations with Price

In [ ]:
numeric_cols = train.select_dtypes(include='number').columns.tolist()
# Remove id-like columns
exclude = ['id', 'host_id', 'host_acceptance_rate']
numeric_cols = [c for c in numeric_cols if c not in exclude]

log_price = np.log1p(train['price'])

correlations = (
    train[numeric_cols]
    .corrwith(log_price)
    .drop('price')
    .sort_values(key=abs, ascending=False)
)

print('Correlation with log1p(price):')
print(correlations.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if v > 0 else 'tomato' for v in correlations]
correlations.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlation with log1p(price)')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

## 5. Borough-Level Price Analysis

In [ ]:
borough_col = 'neighbourhood_group_cleansed'
borough_stats = (
    train.groupby(borough_col)['price']
    .agg(['count', 'mean', 'median'])
    .sort_values('median', ascending=False)
    .round(1)
)
print(borough_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot by borough
boroughs = train[borough_col].dropna().unique()
data_by_borough = [train[train[borough_col] == b]['price'].clip(0, 500) for b in borough_stats.index]
axes[0].boxplot(data_by_borough, labels=borough_stats.index, vert=False)
axes[0].set_title('Price by Borough (clipped at $500)')
axes[0].set_xlabel('Price ($)')

# Listing count by borough
borough_stats['count'].plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Listing Count by Borough')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 6. Room Type and Property Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, ['room_type', 'property_type']):
    top = train[col].value_counts().head(10)
    medians = train.groupby(col)['price'].median().reindex(top.index)
    
    x = range(len(top))
    bars = ax.bar(x, top.values, color='steelblue', alpha=0.7, label='Count')
    ax2 = ax.twinx()
    ax2.plot(x, medians.values, 'o-', color='tomato', label='Median price')
    ax.set_xticks(x)
    ax.set_xticklabels(top.index, rotation=35, ha='right')
    ax.set_ylabel('Count')
    ax2.set_ylabel('Median Price ($)')
    ax.set_title(col)

plt.tight_layout()
plt.show()

## 7. Key Categorical Fields

In [ ]:
for col in ['cancellation_policy', 'bed_type', 'host_response_time']:
    stats = train.groupby(col)['price'].agg(['count', 'median']).sort_values('median', ascending=False)
    print(f'\n--- {col} ---')
    print(stats.to_string())

## 8. Review Scores

In [ ]:
review_cols = [c for c in train.columns if c.startswith('review_scores_')]
print('Review score ranges:')
print(train[review_cols].describe().loc[['count', 'mean', 'min', 'max']].round(2).to_string())

In [ ]:
# Listings with no reviews at all
no_reviews = train['number_of_reviews'] == 0
print(f'Listings with 0 reviews: {no_reviews.sum()} ({no_reviews.mean()*100:.1f}%)')
print(f'  Median price (0 reviews): ${train[no_reviews]["price"].median():.0f}')
print(f'  Median price (has reviews): ${train[~no_reviews]["price"].median():.0f}')

## 9. Text Field Lengths

In [ ]:
text_cols = ['name', 'summary', 'description', 'space', 'neighborhood_overview', 'house_rules']

for col in text_cols:
    lengths = train[col].fillna('').str.len()
    corr = np.corrcoef(lengths, np.log1p(train['price']))[0, 1]
    print(f'{col:30s}  median_len={lengths.median():5.0f}  corr_with_log_price={corr:+.3f}')

## 10. Summary of Key Findings

| Finding | Action |
|---------|--------|
| `price` is right-skewed | Use `log1p(price)` as training target |
| `host_acceptance_rate` 100% missing | Drop this column |
| `square_feet` 99% missing | Add binary flag; impute with 0 |
| Review scores ~23% missing | Add `has_reviews` binary flag; impute with 0 |
| Manhattan >> other boroughs in price | Location is a strong predictor |
| `room_type` strongly separates prices | Encode carefully (ordinal or OHE) |
| Text length weakly correlated with price | Include as a low-cost feature |

In [ ]:
print('Task 1 complete. Proceed to Task 2: src/preprocess.py')